In [0]:
# =============================================================================
# NOTEBOOK  : gold_fact_supplier_performance
# PURPOSE   : silver.purchase_orders → gold.fact_supplier_performance
# LAYER     : Gold (supplier KPI fact)
# FREQUENCY : Daily (after silver_purchase_orders completes)
# WRITE     : Full overwrite — one row per supplier (~200 rows)
#             Always full recompute across entire PO history
# GRAIN     : one row per supplier_id
#
# LOGIC: Aggregate all PO history per supplier. Compute OTD%, lead time
#        variance, quality rating, composite risk score.
#        norm_otd, norm_quality, norm_lt_var are intermediate only —
#        dropped before write, not exposed in PBI schema.
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit
from utils.schema_registry import GOLD_FACT_SUPPLIER_PERFORMANCE_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col, round, when,
    avg, sum as spark_sum, count, stddev,
    datediff, coalesce
)
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
PO_TABLE     = cfg["tables"]["silver_purchase_orders"]
TARGET_TABLE = cfg["tables"]["gold_fact_supplier_performance"]
PIPELINE     = "gold_fact_supplier_performance"

In [0]:
# ── Read + Compute KPIs + Write ──────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, PO_TABLE)
 
try:
    # Full PO history — KPIs always computed across all orders
    po_df = spark.read.table(PO_TABLE)
    rows_read = po_df.count()
    print(f"[READ] {rows_read} purchase order records")
 
    # Derive delivery_delay_days before aggregation
    po_enriched = po_df.withColumn(
        "delivery_delay_days",
        datediff(col("actual_delivery_date"), col("expected_delivery_date"))
    )
 
    # Aggregate KPIs per supplier across full history
    supplier_kpis = (
        po_enriched
        .groupBy("supplier_id")
        .agg(
            round(
                spark_sum(when(col("delivery_delay_days") <= 0, 1).otherwise(0)) /
                count("*") * 100, 2
            ).alias("otd_pct"),
            round(stddev(col("delivery_delay_days").cast("double")), 2).alias("lead_time_variance"),
            round(avg(col("delivery_delay_days").cast("double")),    2).alias("avg_delay_days"),
            round(avg(col("quality_rating").cast("double")),         2).alias("avg_quality_rating"),
            count("*").alias("total_orders"),
            round(spark_sum(col("quantity_ordered").cast("double") * col("unit_cost").cast("double")),2).alias("total_po_value")

        )
    )
 
    # Composite risk score: 40% OTD + 30% quality + 30% lead time consistency
    # Intermediate norm columns computed then dropped — not in output schema
    df = (
        supplier_kpis
        .withColumn("norm_otd",
            col("otd_pct") / 100)
        .withColumn("norm_quality",
            coalesce(col("avg_quality_rating"), lit(0.0)) / 5.0)
        .withColumn("norm_lt_var",
            when(col("lead_time_variance") > 0,
                 lit(1.0) - (col("lead_time_variance") / 10.0))
            .otherwise(lit(1.0)))
        .withColumn("supplier_risk_score",
            round(
                col("norm_otd") * 0.4 +
                col("norm_quality") * 0.3 +
                col("norm_lt_var") * 0.3, 3))
        .withColumn("actual_risk_tier",
            when(col("supplier_risk_score") > 0.55, "Low")
            .when(col("supplier_risk_score") > 0.53, "Medium")
            .otherwise("High"))
        .withColumn("_gold_processed_at", current_timestamp())
        .withColumn("pipeline_run_id",    lit(run_id))
        # norm columns dropped — intermediate only, not in PBI schema
        .select([f.name for f in GOLD_FACT_SUPPLIER_PERFORMANCE_SCHEMA.fields])
    )
 
    rows_written = df.count()
 
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "false")
        .saveAsTable(TARGET_TABLE)
    )
 
    print(f"[DONE] {rows_written} rows written to {TARGET_TABLE}")
 
    end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
              rows_read=rows_read, rows_written=rows_written)
 
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify ───────────────────────────────────────────────────────────
from pyspark.sql.functions import col
 
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()).limit(1) \
    .select("status", "rows_read", "rows_written").show(truncate=False)
 
print("Row count:", spark.read.table(TARGET_TABLE).count())
spark.read.table(TARGET_TABLE) \
    .groupBy("actual_risk_tier").count().show()
spark.read.table(TARGET_TABLE) \
    .orderBy(col("supplier_risk_score").desc()) \
    .select("supplier_id", "otd_pct", "avg_quality_rating",
            "supplier_risk_score", "actual_risk_tier") \
    .show(10, truncate=False)